In [1]:
%%capture
%pip install git+https://github.com/amazon-science/chronos-forecasting.git
%pip install sktime


In [2]:
import os
import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from chronos import ChronosPipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sktime.performance_metrics.forecasting import MeanAbsoluteScaledError


In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Keep the shared dataset config and split used by the comparison notebooks.
CONFIG_PATH = '/content/drive/MyDrive/Models parameters JSON files/seasonalnaive.json'
OUTPUT_DIR = '/content/drive/MyDrive/results/json/Chronos'
TRAIN_RATIO = 0.80
NUM_SAMPLES = 100
SHOW_PLOTS = True

with open(CONFIG_PATH, 'r', encoding='utf-8') as file:
    CONFIG = json.load(file)

DATASETS = CONFIG.get('datasets', [])
if not DATASETS:
    raise ValueError('No datasets were found in the JSON file.')

total_targets = sum(len(dataset.get('targets', [])) for dataset in DATASETS)
if total_targets == 0:
    raise ValueError('No targets were found in the datasets.')

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Loaded {len(DATASETS)} datasets and {total_targets} targets')
print(f'Train ratio: {TRAIN_RATIO:.0%}; samples: {NUM_SAMPLES}')
print(f'Results: {OUTPUT_DIR}')


Mounted at /content/drive
Loaded 6 datasets and 26 targets
Train ratio: 80%; samples: 100
Results: /content/drive/MyDrive/results/json/Chronos


In [4]:
# Keep the same Chronos-T5-small checkpoint for a repeatable comparison.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if device == 'cuda' else torch.float32

pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small', device_map=device, torch_dtype=dtype
)
print('Device:', device)


config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  185MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

Device: cuda


## Helper function: scaled metrics


In [5]:
def scale_values(y_train, y_true, y_pred):
    y_train = np.asarray(y_train, dtype=np.float64).reshape(-1, 1)
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1, 1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1, 1)

    scaler = MinMaxScaler()
    scaler.fit(y_train)
    y_true_scaled = scaler.transform(y_true)
    y_pred_scaled = scaler.transform(y_pred)

    return y_true_scaled, y_pred_scaled


## Helper function: probabilistic metric


In [6]:
def pinball_loss(q, y, alpha):
    return (alpha - (y < q)) * (y - q)


def crps_chronos_quantile_version(samples, y_true):
    samples = np.array(samples)
    y_true = np.array(y_true)

    alphas = np.arange(0.1, 1.0, 0.1)
    q_preds = {
        alpha: np.quantile(samples, alpha, axis=0)
        for alpha in alphas
    }

    denominator = np.sum(np.abs(y_true))
    wql_values = []

    for alpha in alphas:
        q = q_preds[alpha]
        pin = pinball_loss(q, y_true, alpha)
        wql = 2 * np.sum(pin) / denominator
        wql_values.append(wql)

    return float(np.mean(wql_values))


## Calculate forecast metrics


In [7]:
def calculate_metrics(train_values, test_values, predictions, samples):
    test_array = np.asarray(test_values, dtype=np.float64)
    prediction_array = np.asarray(predictions, dtype=np.float64)

    mase = MeanAbsoluteScaledError()(
        test_array, prediction_array, y_train=train_values
    )

    # Unscaled errors use the original values.
    mae = float(mean_absolute_error(test_array, prediction_array))
    mse = float(mean_squared_error(test_array, prediction_array))
    rmse = float(np.sqrt(mse))

    # Scaled errors use a MinMaxScaler fitted only on the training values.
    test_scaled, predictions_scaled = scale_values(
        train_values, test_array, prediction_array
    )
    scaled_mae = float(mean_absolute_error(test_scaled, predictions_scaled))
    scaled_mse = float(mean_squared_error(test_scaled, predictions_scaled))
    scaled_rmse = float(np.sqrt(scaled_mse))

    return {
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'mase': float(mase),
        'mae_scaled': scaled_mae,
        'mse_scaled': scaled_mse,
        'rmse_scaled': scaled_rmse,
        'crps': crps_chronos_quantile_version(samples, test_values),
    }


## Prepare one target column


In [8]:
def prepare_target(data, dataset, target):
    dataset_name = dataset['dataset_name']
    time_column = dataset['time_column']
    target_column = target['column']

    target_data = data[[time_column, target_column]].copy()
    target_data[target_column] = pd.to_numeric(target_data[target_column])

    dates = target_data[time_column]
    values = target_data[target_column].to_numpy(dtype=float)

    return dataset_name, time_column, target_column, dates, values


## Split data into training and test parts


In [9]:
def split_data(dates, values):
    split_index = int(len(values) * TRAIN_RATIO)

    train_dates = dates.iloc[:split_index]
    train_values = values[:split_index]
    test_dates = dates.iloc[split_index:]
    test_values = values[split_index:]

    return train_dates, train_values, test_dates, test_values


## Create a Chronos forecast


In [10]:
def create_forecast(dates, train_values, test_values, num_samples=NUM_SAMPLES):
    # The dates preserve the Lag-Llama interface and align outputs; Chronos predicts
    # from the ordered target values and does not consume dates as model inputs.
    context = torch.tensor(np.asarray(train_values, dtype=np.float32))
    with torch.no_grad():
        forecast = pipeline.predict(
            context,
            prediction_length=len(test_values),
            num_samples=num_samples,
        )

    samples = forecast[0].detach().float().cpu().numpy().astype(np.float64)
    quantile_levels = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    quantile_values = np.quantile(samples, quantile_levels, axis=0)
    quantiles = {
        'p5': quantile_values[0],
        'p10': quantile_values[1],
        'p25': quantile_values[2],
        'median': quantile_values[3],
        'p75': quantile_values[4],
        'p90': quantile_values[5],
        'p95': quantile_values[6],
    }
    return samples, quantiles


## Save full time series to JSON


In [11]:
def save_timeseries_full(
    log_path, model_name, full_dates, full_values, train_dates, train_values,
    test_dates, test_values, forecast_dates, forecast_quantiles, metrics
):
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8') as file:
            log_data = json.load(file)
    else:
        log_data = []

    forecast_data = {'dates': [str(date) for date in forecast_dates]}
    for name, values in forecast_quantiles.items():
        forecast_data[name] = [float(value) for value in values]

    entry = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model': model_name,
        'full_series': {
            'dates': [str(date) for date in full_dates],
            'values': [float(value) for value in full_values],
        },
        'train_series': {
            'dates': [str(date) for date in train_dates],
            'values': [float(value) for value in train_values],
        },
        'test_series': {
            'dates': [str(date) for date in test_dates],
            'values': [float(value) for value in test_values],
        },
        'forecast': forecast_data,
        'metrics': {
            name: None if not np.isfinite(value) else float(value)
            for name, value in metrics.items()
        },
    }

    log_data.append(entry)
    with open(log_path, 'w', encoding='utf-8') as file:
        json.dump(log_data, file, indent=4, ensure_ascii=False)


## Save one result as JSON


In [12]:
def save_result(dataset_name, target_column, target, dates, values,
                train_dates, train_values, test_dates, test_values,
                quantiles, metrics):
    output_file = target.get(
        'output_file', f'{dataset_name}_{target_column}.json'
    )
    output_file = output_file.replace('/', '_').replace('\\', '_').replace(':', '_')
    log_path = os.path.join(OUTPUT_DIR, output_file)

    save_timeseries_full(
        log_path,
        'Chronos-T5-small',
        dates,
        values,
        train_dates,
        train_values,
        test_dates,
        test_values,
        test_dates,
        quantiles,
        metrics,
    )

    result = {
        'dataset': dataset_name,
        'target': target_column,
        'model': 'Chronos-T5-small',
        'output_file': output_file,
    }
    result.update(metrics)
    all_results.append(result)

    print('JSON saved:', log_path)


## Draw the forecast chart


In [13]:
def show_forecast(train_dates, train_values, test_dates, test_values, quantiles,
                  dataset_name, time_column, target_column):
    if not SHOW_PLOTS:
        return

    plt.figure(figsize=(16, 6))
    plt.plot(train_dates, train_values, label='Training data')
    plt.plot(test_dates, test_values, label='Actual values')
    plt.plot(test_dates, quantiles['median'], '--', label='Lag-Llama median')
    plt.fill_between(test_dates, quantiles['p25'], quantiles['p75'], alpha=0.25, label='50% interval')
    plt.fill_between(test_dates, quantiles['p10'], quantiles['p90'], alpha=0.15, label='80% interval')
    plt.title(f'{dataset_name} - {target_column}')
    plt.xlabel(time_column)
    plt.ylabel(target_column)
    plt.legend()
    plt.grid()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    plt.close()


## Run all steps for one target column


In [14]:
def run_target(data, dataset, target, model_number):
    dataset_name, time_column, target_column, dates, values = prepare_target(
        data, dataset, target
    )

    train_dates, train_values, test_dates, test_values = split_data(dates, values)
    samples, quantiles = create_forecast(dates, train_values, test_values)
    predictions = quantiles['median']
    metrics = calculate_metrics(train_values, test_values, predictions, samples)

    print(f"\nModel {model_number}/{total_models}: {dataset_name} - {target_column}")
    print('Scaled MAE:', round(metrics['mae_scaled'], 4))
    print('Scaled RMSE:', round(metrics['rmse_scaled'], 4))
    print('MASE:', round(metrics['mase'], 4))

    show_forecast(
        train_dates, train_values, test_dates, test_values, quantiles,
        dataset_name, time_column, target_column,
    )

    save_result(
        dataset_name, target_column, target, dates, values,
        train_dates, train_values, test_dates, test_values, quantiles, metrics,
    )


## Run all datasets and target columns


In [15]:
total_models = sum(len(dataset['targets']) for dataset in DATASETS)
all_results = []

for dataset in DATASETS:
    dataset_name = dataset['dataset_name']
    file_path = dataset['file_path']
    time_column = dataset['time_column']

    data = pd.read_csv(file_path)
    data[time_column] = pd.to_datetime(data[time_column])
    if data[time_column].isna().any():
        raise ValueError(f"Invalid or missing dates in '{time_column}' for {file_path}")
    data = data.sort_values(time_column).reset_index(drop=True)

    print('\n' + '=' * 60)
    print('Dataset:', dataset_name)

    for target in dataset['targets']:
        run_target(data, dataset, target, len(all_results) + 1)


Output hidden; open in https://colab.research.google.com to view.

## Print completion summary


In [16]:
print('\n\n')
print('=' * 110)
print('ALL DATASETS / TARGETS COMPLETED')
print('=' * 110)
if len(all_results) > 0:
    summary_df = pd.DataFrame(all_results)
    print(summary_df.to_string(index=False))
else:
    print('No target was completed.')

print('\n')
print('=' * 90)
print('PROCESS FINISHED')
print('=' * 90)
print('Datasets requested:', len(DATASETS))
print('Targets requested:', total_models)
print('Targets completed:', len(all_results))
print('Output directory:', OUTPUT_DIR)
if len(all_results) != total_models:
    print('\nWARNING:')
    print('Not all requested targets completed successfully.')
    print('Requested:', total_models)
    print('Completed:', len(all_results))
else:
    print('ALL REQUESTED TARGETS COMPLETED.')





ALL DATASETS / TARGETS COMPLETED
dataset         target            model               output_file           mae          mse          rmse      mase  mae_scaled  mse_scaled  rmse_scaled     crps
   WB90             wt Chronos-T5-small                WB_wt.json      0.072720 2.818120e-02      0.167873  0.541086    0.026735    0.003809     0.061718 0.002312
   WB90            toc Chronos-T5-small               WB_toc.json      0.035410 2.433403e-03      0.049330  0.413771    0.075341    0.011016     0.104956 0.006467
   WB90    temperature Chronos-T5-small       WB_temperature.json      0.805211 9.113361e-01      0.954639  1.422504    0.084848    0.010119     0.100594 0.025406
   WB90             ph Chronos-T5-small                WB_ph.json      0.050245 6.584376e-03      0.081144  0.701904    0.157016    0.064301     0.253576 0.005374
   WB90       humidity Chronos-T5-small          WB_humidity.json      8.051382 9.474938e+01      9.733929  3.405735    0.171306    0.042892     0.20